# Hermite Polynomial Offline Calculator (Linear Fitting for Nonlinear Regression)

**Step 1: Choose a convention (two weight functions and orthogonal bases)**

| Convention | Weight function | Orthogonal basis | Recurrence |
|---|---|---|---|
| Physicists' | $f(x)=e^{-x^2}$ | Physicists' $\{H_n\}$ | $H_0=1,\ H_1=2x,\ H_{n+1}=2xH_n-2nH_{n-1}$ |
| Probabilists' | $\varphi(x)=\dfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$ (standard normal density) | Probabilists' $\{He_n\}$ | $He_0=1,\ He_1=x,\ He_{n+1}=xHe_n-nHe_{n-1}$ |

The two bases are related by $He_n(x)=2^{-n/2}H_n\!\left(\dfrac{x}{\sqrt2}\right)$ and reconstruct the same function.

**Standard terminology**

- Inner product: Physicists' $\langle p,q\rangle_f=\displaystyle\int_{-\infty}^{\infty}p(x)q(x)e^{-x^2}\,dx$;
  Probabilists' $\langle p,q\rangle_\varphi=\displaystyle\int_{-\infty}^{\infty}p(x)q(x)\varphi(x)\,dx$
- What you called the "coefficient basis of the function vector" is formally the **Hermite expansion coefficients (generalized Fourier coefficients)**:
  $\alpha_n=\dfrac{\langle h,P_n\rangle_w}{\langle P_n,P_n\rangle_w}$, so that $h(x)=\displaystyle\sum_{n=0}^{N}\alpha_n P_n(x)$
  ($P_n$ is $H_n$ or $He_n$ depending on the convention)
- Rodrigues notation: Physicists' $\dfrac{f^{(n)}}{f}=(-1)^n H_n$;
  Probabilists' $\dfrac{\varphi^{(n)}}{\varphi}=(-1)^n He_n$; the coefficient vector in this basis is also provided

**Two modes (two tabs)**

| | Input | Cell | Method |
|---|---|---|---|
| Mode 1 | $h$ known: input degree $d$ and coefficients $a_k$ | Cell 4 | Exact rational arithmetic; outputs $\alpha_n$ and coefficient vectors as fractions |
| Mode 2 | $h$ unknown: input $m$ points $(x_i,y_i)$ and max order $N$ | Cell 5 | Trapezoidal numerical integration per formula III; outputs the full polynomial |

**Mode 2 stopping metric $\Delta_n$: computation principle**

- Partial sum: $P_n(x)=\displaystyle\sum_{j=0}^{n}\alpha_jB_j(x)$ (with $B_j=H_j$ or $He_j$ by convention); the true function is $\varphi=\lim_{n\to\infty}P_n$, and the fitted polynomial currently returned by the tool is $F=P_N$.
- Definition (relative weighted-norm increment between two successive partial sums; $n\ge1$, $\Delta_0$ undefined):

$$\Delta_n=\frac{\|P_n-P_{n-1}\|_w}{\|P_n\|_w}
=\sqrt{\frac{\displaystyle\int (P_n-P_{n-1})^2w(x)\,dx}{\displaystyle\int P_n^2(x)w(x)\,dx}}
=\frac{\|\alpha_nB_n\|_w}{\|P_n\|_w},\qquad P_n-P_{n-1}=\alpha_nB_n$$

- Expanding the squared difference reveals the **product (unsquared) cross term** $P_nP_{n-1}$ that appears in the instructor's red-pen board work:

$$\int (P_n-P_{n-1})^2w\,dx
=\int P_n^2w\,dx-2\int P_n(x)P_{n-1}(x)w(x)\,dx+\int P_{n-1}^2w\,dx$$

- Board comparison (important): the blue-pen norm definition and the squared-difference integral above are the standard convention. In the worked-example area the instructor's red "i.e." briefly wrote a product-root form $\sqrt{\int P_nP_{n-1}w\,dx\big/\int P_n^2w\,dx}$. The standalone $\int P_nP_{n-1}w=\langle P_n,P_{n-1}\rangle_w$ is only the cross inner-product term in the expansion, **not** the numerator $\|P_n-P_{n-1}\|_w^2$; under exact orthogonality it equals $\|P_{n-1}\|_w^2$, so that literal ratio tends to 1 as $n$ grows and cannot measure the "change between orders". On the default 6-point homework data (physicists' convention, node trapezoid) the literal product form gives roughly $0.722/0.971/0.998/0.997/1.001$, which does not match the board's left-column values $0.506050/0.123712/0.092488/0.042527/0.021085$; the third column therefore uses the squared-difference norm convention that reproduces every board value. Please confirm with the instructor whether the red line was the expanded form or omitted $(P_n-P_{n-1})^2$.
- Discrete implementation: both weighted squared integrals (numerator and denominator) are evaluated with the **trapezoidal rule on the data nodes** (not the analytic full-axis norm); Cell 5 shows the continuous integral form, the full discrete formula and the rowwise values.
- Stopping rule: the classroom threshold suggests stopping when $\Delta_N<0.05$ and taking $\varphi\approx P_N$. Note $\Delta_n$ is the **norm ratio after the square root**; $\Delta_n^2$ is the squared-energy ratio. In the red underdetermined case (fewer points than parameters) the stop signal is demonstration-only and must be combined with SSE, the residual table and the three-tier data warning.

All formulas are typeset in LaTeX with stacked fractions; fractions in Mode 2 are rational approximations of decimals (denominator capped at 10,000).

In [1]:
# ===== Cell 2 | Dependency imports and global configuration (no changes needed, run as-is) =====
import re
import numpy as np
import sympy as sp
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, Latex, Markdown, clear_output

x = sp.symbols('x')
%matplotlib inline

In [2]:
# ===== Cell 3 | Core math functions (parameterized for both conventions, no changes needed) =====

# ---------- Convention config: a is the recurrence scale (2 for physicists, 1 for probabilists) ----------
CONV = {
    'phys': {
        'a': 2,
        'weight_num': lambda xv: np.exp(-xv ** 2),
        'bname': 'H', 'wname': 'f',
        'inner_tex': r'\int_{-\infty}^{\infty}p(x)q(x)e^{-x^2}\,dx',
        'weight_tex': r'e^{-x^2}',
    },
    'prob': {
        'a': 1,
        'weight_num': lambda xv: np.exp(-xv ** 2 / 2.0),
        'bname': 'He', 'wname': r'\varphi',
        'inner_tex': r'\int_{-\infty}^{\infty}p(x)q(x)\frac{1}{\sqrt{2\pi}}e^{-x^2/2}\,dx',
        'weight_tex': r'\frac{1}{\sqrt{2\pi}}e^{-x^2/2}',
    },
}
CONV_NAME = 'phys'

# ---------- Exact mode ----------
def parse_rational(s):
    """Parse integers, decimals and fractions (e.g. 7/2, -1/4) into exact rationals sp.Rational"""
    s = str(s).strip().replace(' ', '')
    if '/' in s:
        p, q = s.split('/')
        return sp.Rational(int(p), int(q))
    return sp.Rational(s)

def exact_basis_poly(n, conv_name):
    """Build the nth basis polynomial exactly via recurrence P_{k+1} = a x P_k - a k P_{k-1}"""
    a = sp.Integer(CONV[conv_name]['a'])
    if n == 0:
        return sp.Integer(1)
    p_prev, p_cur = sp.Integer(1), a * x
    for k in range(1, n):
        p_prev, p_cur = p_cur, sp.expand(a * x * p_cur - a * k * p_prev)
    return p_cur

def basis_matrix(d, conv_name):
    """M[k,n] = coefficient of x^k in P_n; ordinary coefficients a and expansion coefficients alpha satisfy a = M alpha"""
    M = sp.zeros(d + 1, d + 1)
    for n in range(d + 1):
        for monom, c in sp.Poly(exact_basis_poly(n, conv_name), x).terms():
            M[monom[0], n] = c
    return M

def exact_expansion(coeffs, conv_name):
    """Input ascending ordinary coefficients [a0,...,ad]; return alpha (P_n basis), beta (Rodrigues basis), h expression"""
    d = len(coeffs) - 1
    alpha = [sp.simplify(v) for v in basis_matrix(d, conv_name).solve(sp.Matrix(coeffs))]
    beta = [((-1) ** n) * v for n, v in enumerate(alpha)]
    h_expr = sp.expand(sum(coeffs[k] * x ** k for k in range(d + 1)))
    return alpha, beta, h_expr

# ---------- Numeric mode ----------
def basis_coeff_arrays(N, conv_name):
    """Build ascending coefficient arrays of P_0...P_N (float)"""
    a = float(CONV[conv_name]['a'])
    P = [np.array([1.0]), np.array([0.0, a])]
    for n in range(1, N):
        shifted = np.concatenate(([0.0], a * P[n]))            # a x P_n
        prev_pad = np.zeros(len(shifted))
        prev_pad[:len(P[n - 1])] = P[n - 1]
        P.append(shifted - a * n * prev_pad)
    return P[:N + 1]

def basis_values(xv, N, conv_name):
    """Values of P_n on the nodes xv; returns an (N+1) x m array"""
    arrays = basis_coeff_arrays(N, conv_name)
    return np.vstack([np.polyval(p[::-1], xv) for p in arrays])

def trapezoid_alpha(xv, yv, N, conv_name):
    """Formula III: numerically compute alpha_0...alpha_N by trapezoidal rule over the node interval [x_1,x_m]"""
    Pv = basis_values(xv, N, conv_name)
    weight = CONV[conv_name]['weight_num'](xv)
    dx = np.diff(xv)
    alpha = []
    for n in range(N + 1):
        g_num = yv * Pv[n] * weight
        g_den = Pv[n] ** 2 * weight
        num = np.sum(dx / 2.0 * (g_num[1:] + g_num[:-1]))
        den = np.sum(dx / 2.0 * (g_den[1:] + g_den[:-1]))
        alpha.append(num / den)
    return np.array(alpha, dtype=float), Pv

def trapezoid_delta(xv, Pv, alpha, conv_name):
    """Classroom Truncation Rule (whiteboard convention):
    Delta_n = ||P_n-P_{n-1}||_w / ||P_n||_w, both norms evaluated by the trapezoidal
    rule on the data nodes. Here P_n on the left denotes the partial sum;
    P_n-P_{n-1} = alpha_n*B_n(x), where B_n=H_n/He_n is the n-th basis polynomial
    (passed in as Pv[n], NOT the partial sum P_n). Returns length N+1; delta[0]=nan."""
    weight = CONV[conv_name]['weight_num'](xv)
    dx = np.diff(xv)
    def _trap(g):
        return float(np.sum(dx / 2.0 * (g[1:] + g[:-1])))
    N = len(alpha) - 1
    out = np.full(N + 1, np.nan)
    Pn_nodes = np.zeros_like(xv, dtype=float)
    for n in range(N + 1):
        Pn_nodes = Pn_nodes + alpha[n] * Pv[n]
        if n >= 1:
            dP = alpha[n] * Pv[n]
            num = _trap(dP ** 2 * weight)
            den = _trap(Pn_nodes ** 2 * weight)
            out[n] = float(np.sqrt(num / den)) if den > 0 else np.nan
    return out

def expand_numeric(alpha, conv_name):
    """Merge sum alpha_n P_n into ascending ordinary-polynomial coefficients"""
    arrays = basis_coeff_arrays(len(alpha) - 1, conv_name)
    c = np.zeros(max(len(p) for p in arrays))
    for n, a in enumerate(alpha):
        c[:len(arrays[n])] += a * arrays[n]
    return c

def approx_frac(v, max_den=10000):
    """Convert a decimal to an approximate fraction for LaTeX display"""
    return sp.Rational(float(v)).limit_denominator(max_den)

def _in_fixed_range(v):
    """Fixed-range check: fixed decimals and approximate fractions are shown only in this range"""
    av = abs(float(v))
    return v != 0 and av < 1e5 and av >= 1e-4

def _sci_tex(v):
    """LaTeX form of scientific notation, e.g. 2.711\\times 10^{4}"""
    mant, exp = ('%.3e' % float(v)).split('e')
    return r'%s\times 10^{%d}' % (mant, int(exp))

def fmt_num(v, fixed=6, latex=False):
    """Fixed decimals in the normal range; scientific notation (3 significant digits) for large (>=1e5) or small (<1e-4) values"""
    if float(v) == 0:
        return '0'
    if not _in_fixed_range(v):
        return _sci_tex(v) if latex else '%.3e' % float(v)
    return ('%.' + str(fixed) + 'f') % float(v)

def _coef_parts(v, approx=True):
    """Split a coefficient into (negative sign, body LaTeX); approximate fractions in range, scientific notation outside; None for zero"""
    if float(v) == 0:
        return None
    if _in_fixed_range(v):
        c = approx_frac(v) if approx else sp.simplify(v)
        return _split_sign(c)
    return float(v) < 0, fmt_num(abs(float(v)), latex=True)

def L(s):
    """Wrap bare LaTeX in a gathered environment so it renders as a formula rather than source text"""
    return Latex(r'\begin{gathered} ' + s + r' \end{gathered}')

def _split_sign(c):
    """Split out the sign and body LaTeX of a coefficient, avoiding double spaces from string replacement"""
    s = sp.latex(c)
    if s.startswith('-'):
        return True, s[1:].lstrip()
    return False, s

def _join_signed(signed_terms):
    """signed_terms is a list of (negative sign, body LaTeX); join with proper plus/minus signs"""
    if not signed_terms:
        return '0'
    neg, body = signed_terms[0]
    s = '-' + body if neg else body
    for neg, body in signed_terms[1:]:
        s += (' - ' if neg else ' + ') + body
    return s

def latex_basis_form(alpha, conv_name, approx=True):
    """Convert the alpha sequence to LaTeX of sum alpha_n P_n"""
    bn = CONV[conv_name]['bname']
    signed = []
    for n, v in enumerate(alpha):
        parts = _coef_parts(v, approx)
        if parts is None:
            continue
        neg, body = parts
        signed.append((neg, body + r'\,%s_{%d}(x)' % (bn, n)))
    return _join_signed(signed)

def latex_standard_poly(c_asc, approx=True):
    """Convert ascending coefficients to standard-polynomial LaTeX"""
    signed = []
    for k, v in enumerate(c_asc):
        parts = _coef_parts(v, approx)
        if parts is None:
            continue
        neg, body = parts
        if k == 0:
            tail = ''
        elif k == 1:
            tail = r'\,x'
        else:
            tail = r'\,x^{%d}' % k
        signed.append((neg, body + tail))
    return _join_signed(signed)

def rodrigues_tex(conv_name):
    """LaTeX of the Rodrigues identity"""
    if conv_name == 'phys':
        return r'\frac{f^{(n)}}{f}=(-1)^n H_n'
    return r'\frac{\varphi^{(n)}}{\varphi}=(-1)^n He_n'

def static_trap_tex(conv_name):
    """Full LaTeX of formula III (basis symbol and weight function depend on the convention)"""
    bn = CONV[conv_name]['bname']
    if conv_name == 'phys':
        w = lambda i: r'e^{-x_{%s}^2}' % i
    else:
        w = lambda i: r'\frac{1}{\sqrt{2\pi}}e^{-x_{%s}^2/2}' % i
    return (
        r'\alpha_n \approx \frac{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[y_{i+1}%s_n(x_{i+1})%s+y_i%s_n(x_i)%s\right]}{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[%s_n^2(x_{i+1})%s+%s_n^2(x_i)%s\right]}'
        % (bn, w('i+1'), bn, w('i'), bn, w('i+1'), bn, w('i')))

def static_delta_tex(conv_name):
    """Node-trapezoid discrete formula for the stopping metric Delta_n
    (trapezoidal summation on the data nodes)."""
    bn = CONV[conv_name]['bname']
    if conv_name == 'phys':
        w = lambda i: r'e^{-x_{%s}^2}' % i
    else:
        w = lambda i: r'\frac{1}{\sqrt{2\pi}}e^{-x_{%s}^2/2}' % i
    w1, w0 = w('i+1'), w('i')
    return (
        r'\Delta_n \approx \sqrt{\frac{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[(\alpha_n %s_n(x_{i+1}))^2%s+(\alpha_n %s_n(x_i))^2%s\right]}{'
        r'\sum_{i=1}^{m-1}\frac{x_{i+1}-x_i}{2}'
        r'\left[P_n(x_{i+1})^2%s+P_n(x_i)^2%s\right]}}'
        % (bn, w1, bn, w0, w1, w0))

def order_warning(m, N):
    """Three-tier order warning: returns (level, text) with level ok / caution / danger (p=N+1 parameters)"""
    p = N + 1
    if m < p:
        return 'danger', ('Underdetermined: %d parameters exceed %d points, so the solution is not unique. '
                          'The results below are for demonstration only and carry no statistical meaning' % (p, m))
    if m < 2 * p:
        return 'caution', ('Estimating %d coefficients from %d points: fewer than 2 points per parameter, '
                           'so coefficients may be unstable; cite with caution' % (m, p))
    return 'ok', ''

_WARN_STYLE = {
    'caution': ('#fef3c7', '#f59e0b', '#92400e'),
    'danger': ('#fee2e2', '#ef4444', '#991b1b'),
}

def warning_banner(m, N):
    """Build the yellow/red warning banner as Markdown; returns an empty string for the ok level"""
    level, text = order_warning(m, N)
    if level == 'ok':
        return ''
    bg, border, color = _WARN_STYLE[level]
    return Markdown(
        '<div style="background:%s;border:1px solid %s;color:%s;border-radius:10px;'
        'padding:10px 14px;margin:10px 0;font-size:13.5px">%s</div>'
        % (bg, border, color, text))

def stop_banner(delta_arr, N, warn_level='ok', threshold=0.05):
    """Classroom Truncation Rule hint: when the current highest order satisfies
    Delta_N<threshold show a green 'safe to stop' banner; downgrade to amber in the
    underdetermined danger case; otherwise show a neutral blue 'keep increasing' note."""
    if (N < 1 or delta_arr is None or N >= len(delta_arr)
            or not np.isfinite(delta_arr[N])):
        return ''
    dN = float(delta_arr[N])
    first = next((n for n in range(1, N + 1)
                  if np.isfinite(delta_arr[n]) and delta_arr[n] < threshold), None)
    cross = ('' if first is None or first == N else
             ' Within the current range 1-%d, Delta_n first dropped below %.2f at n=%d; '
             'with sparse nodes high orders can rebound, so when adding orders one by one, '
             'prefer two consecutive orders below the threshold before stopping.'
             % (N, threshold, first))
    if dN < threshold:
        if warn_level == 'danger':
            bg, bd, fg = '#fef3c7', '#f59e0b', '#92400e'
            head = ('Delta_%d = %.6f &lt; %.2f, formally meeting the classroom stopping threshold; '
                    'however this is the red underdetermined case (more parameters than points), so the '
                    'signal is unstable and results are for demonstration only; add data if possible.'
                    % (N, dN, threshold))
        else:
            bg, bd, fg = '#dcfce7', '#22c55e', '#166534'
            head = ('Delta_%d = %.6f &lt; %.2f: the relative weighted-norm change between the last two '
                    'partial sums is below 5%%. Per the classroom Truncation Rule (stability of P_n), you may '
                    'stop at this order and take phi ~ P_%d; also check SSE, the residual table and the data-warning banner.'
                    % (N, dN, threshold, N))
    else:
        bg, bd, fg = '#eff6ff', '#3b82f6', '#1e3a8a'
        head = ('Delta_%d = %.6f >= %.2f: the order-%d increment is still at least 5%% in relative norm; '
                'per the stability rule you may keep increasing the order and watch the trend. If a yellow/red '
                'data warning, an SSE rebound or high-order oscillation also appears, judge jointly rather than '
                'blindly adding orders.' % (N, dN, threshold, N))
    return Markdown(
        '<div style="background:%s;border:1px solid %s;color:%s;border-radius:10px;'
        'padding:10px 14px;margin:10px 0;font-size:13.5px;line-height:1.6">%s%s</div>'
        % (bg, bd, fg, head, cross))

In [3]:
# ===== Cell 4 | Mode 1: h(x) known; input degree d and coefficients a_k, then run this cell to show the interface =====
conv_select1 = widgets.ToggleButtons(
    options=[("Physicists' H_n (weight f=e^{-x²})", 'phys'),
             ("Probabilists' He_n (weight φ = standard normal)", 'prob')],
    value=CONV_NAME, description='Convention', style={'description_width': 'initial'})

def _on_conv1(change):
    global CONV_NAME
    CONV_NAME = change['new']
    if 'conv_select2' in globals() and conv_select2.value != change['new']:
        conv_select2.value = change['new']

conv_select1.observe(_on_conv1, names='value')

degree_box = widgets.BoundedIntText(value=2, min=0, max=12, description='Degree d')
coeff_box = widgets.VBox()

def rebuild_coeff_inputs(_=None):
    coeff_box.children = [
        widgets.HBox([
            widgets.Label('a_%d (coefficient of x^%d): ' % (k, k)),
            widgets.Text(value='0', layout=widgets.Layout(width='150px'))
        ]) for k in range(degree_box.value + 1)
    ]

degree_box.observe(rebuild_coeff_inputs, names='value')
rebuild_coeff_inputs()
mode1_out = widgets.Output(layout=widgets.Layout(
    max_height='560px', overflow='auto', border='1px solid #e5e7eb',
    padding='8px 10px', margin='8px 0'))

def fill_mode1(vals):
    """One-click example fill: vals is a list of ascending coefficient strings"""
    degree_box.value = len(vals) - 1
    rebuild_coeff_inputs()
    for k, v in enumerate(vals):
        coeff_box.children[k].children[1].value = v

def run_mode1(_):
    with mode1_out:
        clear_output(wait=True)
        try:
            coeffs = [parse_rational(row.children[1].value) for row in coeff_box.children]
            if all(c == 0 for c in coeffs):
                raise ValueError('Coefficients cannot all be zero')
            conv = conv_select1.value
            cfg = CONV[conv]
            bn, wn = cfg['bname'], cfg['wname']
            alpha, beta, h_expr = exact_expansion(coeffs, conv)
            d = len(coeffs) - 1
            display(L(r'Input function: \quad h(x) = ' + sp.latex(h_expr)))
            basis = r' \\ '.join(
                r'%s_{%d}(x) = %s' % (bn, n, sp.latex(exact_basis_poly(n, conv)))
                for n in range(d + 1))
            display(Latex(r'Orthogonal basis used: $\begin{align*} %s \end{align*}$' % basis))
            display(L(
                r'\alpha_n = \frac{\langle h,%s_n\rangle_{%s}}{\langle %s_n,%s_n\rangle_{%s}},'
                r'\qquad \langle p,q\rangle_{%s} = %s'
                % (bn, wn, bn, bn, wn, wn, cfg['inner_tex'])))
            lines = r' \\ '.join(
                r'\alpha_{%d} = %s' % (n, sp.latex(alpha[n])) for n in range(d + 1))
            display(Latex(
                r'Hermite expansion coefficients (generalized Fourier coefficients): $\begin{align*} %s \end{align*}$' % lines))
            display(L(r'\therefore\quad h(x) = ' + latex_basis_form(alpha, conv, approx=False)))
            avec = r' \\ '.join(sp.latex(v) for v in alpha)
            bvec = r' \\ '.join(sp.latex(v) for v in beta)
            display(L(
                r'Coefficient vectors: \quad F_{%s}=\begin{bmatrix} %s \end{bmatrix}\quad \text{(basis %s_n)},'
                r'\qquad %s,\ F=\begin{bmatrix} %s \end{bmatrix}'
                % (bn, avec, bn, rodrigues_tex(conv), bvec)))
        except Exception as e:
            display(Markdown('<span style="color:#c0392b">Invalid input: %s</span>' % e))

btn1 = widgets.Button(description='Compute Hermite expansion', button_style='primary')
btn1.on_click(run_mode1)
demo1a = widgets.Button(description='Example: x²+2x+3')
demo1a.on_click(lambda _: (fill_mode1(['3', '2', '1']), run_mode1(None)))
demo1b = widgets.Button(description='Example: 9x⁴-7x³+6x²-11x+3')
demo1b.on_click(lambda _: (fill_mode1(['3', '-11', '6', '-7', '9']), run_mode1(None)))

mode1_ui = widgets.VBox([
    conv_select1,
    widgets.Label('h(x) = a_0 + a_1 x + … + a_d x^d; coefficients may be integers, decimals or fractions (e.g. 7/2)'),
    degree_box,
    coeff_box,
    widgets.HBox([btn1, demo1a, demo1b]),
    mode1_out
])

In [4]:
# ===== Cell 5 | Mode 2: h unknown; input m points and max order N, then run this cell to show the interface =====
conv_select2 = widgets.ToggleButtons(
    options=[("Physicists' H_n (weight f=e^{-x²})", 'phys'),
             ("Probabilists' He_n (weight φ = standard normal)", 'prob')],
    value=CONV_NAME, description='Convention', style={'description_width': 'initial'})

def _on_conv2(change):
    global CONV_NAME
    CONV_NAME = change['new']
    if 'conv_select1' in globals() and conv_select1.value != change['new']:
        conv_select1.value = change['new']

conv_select2.observe(_on_conv2, names='value')

demo_points = ('0.13, 2.417\n0.47, 3.186\n1.26, 4.732\n'
               '1.83, 4.105\n3.14, 2.891\n4.72, 1.634')
points_area = widgets.Textarea(value=demo_points,
                               layout=widgets.Layout(width='340px', height='190px'))
order_box = widgets.BoundedIntText(value=4, min=0, max=20, description='Max order N')
order_hint2 = widgets.HTML(
    '<div style="font-size:12.5px;color:#6b7280;line-height:1.6;max-width:760px;margin:2px 0 6px">'
    'This classroom-demo version currently supports <b>N≤20</b>. Higher orders (e.g. N=1000) produce very long '
    'coefficient tables and polynomials, and floating-point Hermite recursion may overflow or oscillate. '
    'Supporting N=1000 later would require paged/virtual rendering, result export and more stable numerics. '
    'For large datasets, the output area and the residual text area (above 1000 rows) scroll internally.</div>')
mode2_out = widgets.Output(layout=widgets.Layout(
    max_height='580px', overflow='auto', border='1px solid #e5e7eb',
    padding='8px 10px', margin='8px 0'))
prog2_label = widgets.HTML(value='')
prog2 = widgets.IntProgress(value=0, min=0, max=100, description='0%', bar_style='info',
                            style={'bar_color': '#2563eb'},
                            layout=widgets.Layout(width='100%', max_width='760px'))
prog2_box = widgets.VBox([prog2_label, prog2],
                         layout=widgets.Layout(display='none', margin='6px 0', max_width='760px'))

def set_prog2(value, label, bar_style='info'):
    prog2_box.layout.display = ''
    prog2.value = int(max(0, min(100, value)))
    prog2.description = '%d%%' % prog2.value
    prog2.bar_style = bar_style
    prog2_label.value = '<span style="font-size:13px;color:#4b5563">%s</span>' % label

def parse_points(text):
    """Parse one point per line: comma or whitespace separated; automatically sorted by x"""
    pts = []
    for line in text.strip().splitlines():
        line = line.replace('，', ',').strip()
        if not line:
            continue
        parts = re.split(r'[,\s]+', line)
        if len(parts) != 2:
            raise ValueError('Unparseable line: “%s”; each line needs two numbers x, y' % line)
        pts.append((float(parts[0]), float(parts[1])))
    if len(pts) < 2:
        raise ValueError('At least 2 points are required')
    pts.sort(key=lambda t: t[0])
    xv = np.array([p[0] for p in pts])
    if len(set(xv.tolist())) != len(pts):
        raise ValueError('Duplicate x values are not allowed')
    return xv, np.array([p[1] for p in pts])

def run_mode2(_):
    with mode2_out:
        clear_output(wait=True)
        try:
            set_prog2(2, 'Parsing input points…')
            xv, yv = parse_points(points_area.value)
            N = int(order_box.value)
            m = len(xv)
            if N > 20:
                raise ValueError('This classroom-demo version currently supports N<=20; higher orders create very long output and unstable high-order numerics. N=1000 would require paged/virtual rendering and export support.')
            conv = conv_select2.value
            level, _ = order_warning(m, N)
            banner = warning_banner(m, N)
            if banner:
                display(banner)
            set_prog2(12, 'Computing αₙ by trapezoidal integration…')
            alpha, Pv = trapezoid_alpha(xv, yv, N, conv)
            set_prog2(38, 'Computing Δₙ stability increments…')
            delta_arr = trapezoid_delta(xv, Pv, alpha, conv)
            set_prog2(55, 'Combining polynomial and fitted values…')
            c = expand_numeric(alpha, conv)
            yfit = sum(alpha[n] * Pv[n] for n in range(N + 1))
            set_prog2(68, 'Preparing coefficients, SSE and polynomial forms…')
            display(L(static_trap_tex(conv)))
            display(Markdown('Numerical integration approximates the full-space integral over [%.3g, %.3g]; %d points, %d intervals.'
                             % (xv[0], xv[-1], m, m - 1)))
            display(Markdown(
                r'**Computation principle of the stopping metric $\Delta_n$**. '
                r'The instructor’s blue-pen norm definition is '
                r'$\Delta_n=\|P_n-P_{n-1}\|_w/\|P_n\|_w$, where $P_n=\sum_{j=0}^{n}\alpha_jB_j$ and '
                r'$P_n-P_{n-1}=\alpha_nB_n$. In continuous integral form (the theoretical domain is the whole real axis; '
                r'the tool approximates it by the trapezoidal rule on the data interval $[x_1,x_m]$):'
                r'$$\begin{aligned}'
                r'\Delta_n&=\sqrt{\frac{\int_a^b (P_n-P_{n-1})^2w(x)\,dx}{\int_a^b P_n^2(x)w(x)\,dx}}\\'
                r'&=\sqrt{\frac{\int_a^b [P_n^2(x)-2P_n(x)P_{n-1}(x)+P_{n-1}^2(x)]w(x)\,dx}'
                r'{\int_a^b P_n^2(x)w(x)\,dx}}'
                r'\end{aligned}$$'
                r'The second equality expands the square; the middle term '
                r'$-2\int P_n(x)P_{n-1}(x)w(x)\,dx$ is the unsquared product cross term seen in the instructor’s '
                r'red-pen board work. Discretizing both integrals on the data nodes gives the computable formula below, '
                r'and the third column of the coefficient table shows its rowwise values ($n\geq1$; $\Delta_0$ is undefined).'))
            display(L(static_delta_tex(conv)))
            display(Markdown(
                r'**Red-pen “i.e.” board form for comparison (please verify with the instructor)**: the worked-example '
                r'area also shows a product-root form '
                r'$\Delta_n=\sqrt{\int_a^b P_n(x)P_{n-1}(x)w(x)\,dx\ \big/\ \int_a^b P_n^2(x)w(x)\,dx}$. '
                r'The standalone product integral $\int P_nP_{n-1}w=\langle P_n,P_{n-1}\rangle_w$ is not the norm numerator '
                r'$\|P_n-P_{n-1}\|_w^2=\int(P_n-P_{n-1})^2w$ (the expanded square also contains $P_n^2$, $P_{n-1}^2$ and the '
                r'coefficient $-2$); on the default 6-point data its literal evaluation does not reproduce the board values '
                r'$\Delta_1=0.506050,\Delta_2=0.123712,\ldots$. This tool implements the node-trapezoid version of the '
                r'blue-pen norm definition; the estimator can be switched if the instructor confirms the product-root form.'))
            display(Markdown(
                r'Three aligned columns per row: $\alpha_n$ is the expansion coefficient; '
                r'$\Delta\alpha_n=\alpha_n-\alpha_{n-1}$ is its first difference; '
                r'$\Delta_n=\|P_n-P_{n-1}\|_w/\|P_n\|_w$ is the weighted-norm relative increment used by the '
                r'classroom stopping rule (trapezoidal rule on the data nodes, $n\geq1$). A hint appears below '
                r'when the current highest order satisfies $\Delta_N<0.05$.'))
            def alpha_row(n):
                v = alpha[n]
                if _in_fixed_range(v):
                    head = r'\alpha_{%d} &= %s \approx %s' % (
                        n, fmt_num(v), sp.latex(approx_frac(v)))
                else:
                    head = r'\alpha_{%d} &= %s' % (n, fmt_num(v, latex=True))
                if n >= 1:
                    d = float(v) - float(alpha[n - 1])
                    head += r', & \Delta\alpha_{%d} &= %s' % (
                        n, fmt_num(d, latex=True))
                    dn = delta_arr[n]
                    dn_tex = fmt_num(dn, fixed=6, latex=True) if np.isfinite(dn) else r'\text{--}'
                    head += r', & \Delta_{%d} &= %s' % (n, dn_tex)
                return head
            rows = r' \\ '.join(alpha_row(n) for n in range(N + 1))
            display(Latex(r'$\begin{aligned} %s \end{aligned}$' % rows))
            stop = stop_banner(delta_arr, N, level)
            if stop:
                display(stop)
            sse = float(np.sum((yv - yfit) ** 2))
            mse = sse / m
            display(L(
                r'\begin{aligned} '
                r'\mathrm{SSE} &= \sum_{i=1}^{m}(y_i-F(x_i))^2 = ' + fmt_num(sse, latex=True) + r', \\ '
                r'\mathrm{MSE} &= \frac{1}{m}\sum_{i=1}^{m}(y_i-F(x_i))^2 '
                r'= \frac{\mathrm{SSE}}{m} = ' + fmt_num(mse, latex=True) +
                r' \end{aligned}'))
            display(L(r'F(x) = ' + latex_basis_form(alpha, conv)
                      + r'\quad \text{(Hermite form; fractions are approximate)}'))
            display(L(r'F(x) = ' + latex_standard_poly(c)
                      + r'\quad \text{(standard polynomial form; fractions are approximate)}'))
            set_prog2(76, 'Building residual table…')
            if m <= 1000:
                tbl_lines = ['| i | x_i | y_i | F(x_i) | Residual |',
                             '|---|---|---|---|---|']
                for i in range(m):
                    tbl_lines.append('| %d | %.4g | %.4g | %.4g | %.4g |' % (
                        i + 1, xv[i], yv[i], yfit[i], yv[i] - yfit[i]))
                    if (i + 1) % 2000 == 0 or i == m - 1:
                        set_prog2(76 + 16 * (i + 1) / m, 'Residual rows %d/%d' % (i + 1, m))
                display(Markdown('\n'.join(tbl_lines)))
            else:
                display(Markdown('**Residual table (%d rows)**: large datasets use a scrollable text area with tab-separated values; select all to copy into Excel.' % m))
                tsv_lines = ['i\tx_i\ty_i\tF(x_i)\tresidual']
                for i in range(m):
                    tsv_lines.append('%d\t%.6g\t%.6g\t%.6g\t%.6g' % (
                        i + 1, xv[i], yv[i], yfit[i], yv[i] - yfit[i]))
                    if (i + 1) % 2000 == 0 or i == m - 1:
                        set_prog2(76 + 16 * (i + 1) / m, 'Scrollable residual rows %d/%d' % (i + 1, m))
                display(widgets.Textarea(
                    value='\n'.join(tsv_lines), disabled=True,
                    layout=widgets.Layout(height='380px', width='100%', max_width='760px')))
            set_prog2(94, 'Drawing fit plot…')
            grid = np.linspace(xv[0], xv[-1], 400)
            Pg = basis_values(grid, N, conv)
            yg = sum(alpha[n] * Pg[n] for n in range(N + 1))
            plt.figure(figsize=(7, 3.4))
            plt.plot(grid, yg, label='F(x)')
            plt.scatter(xv, yv, color='crimson', zorder=3, label='data points')
            plt.xlabel('x'); plt.ylabel('y'); plt.legend()
            plt.grid(alpha=.3); plt.tight_layout()
            plt.show()
            set_prog2(100, 'Done: %d points, max order N=%d' % (m, N), 'success')
        except Exception as e:
            set_prog2(prog2.value, 'Stopped with an error: %s' % e, 'danger')
            display(Markdown('<span style="color:#c0392b">Invalid input: %s</span>' % e))

# ----- Mode 2 random-point generator (fills the textarea only; no auto-run) -----
gen_count = widgets.Text(value="", placeholder="Required, e.g. 20", description="Point count",
                         style={"description_width": "initial"})
gen_range_on = widgets.Checkbox(
    value=False,
    description="Limit coordinate range (when checked, one uniform interval applies to both x and y)",
    style={"description_width": "initial"}, layout=widgets.Layout(width="auto"))
gen_min = widgets.FloatText(value=-2.0, description="Min",
                            style={"description_width": "initial"})
gen_max = widgets.FloatText(value=3.0, description="Max",
                            style={"description_width": "initial"})
gen_order = widgets.Dropdown(options=[("Unsorted (random order)", "none"),
                                      ("Sort x ascending", "asc"),
                                      ("Sort x descending", "desc")],
                             value="asc", description="Order",
                             style={"description_width": "initial"})
gen_btn = widgets.Button(description="Generate random points", button_style="info")
gen_msg = widgets.HTML(value="")
gen_hint = widgets.HTML(
    '<div style="font-size:12.5px;color:#6b7280;line-height:1.6;max-width:690px">'
    "Point count range: 2–20000. Range off: both x and y are sampled from the standard normal N(0,1) (no hard bounds). "
    "Range on: both x and y are uniform on [Min, Max], and Min must be strictly smaller than Max. "
    "Changing Min or Max automatically checks the range box. "
    "Descending order only changes how lines are placed in the textarea; after clicking Compute, "
    "parse_points still sorts by x ascending. Generated text uses 17 significant digits and guarantees distinct x values, "
    "preventing 20,000 points from colliding after rounding. Click “Compute full polynomial” after generation.</div>")

def _toggle_gen_range(change):
    if not gen_range_on.value:
        gen_msg.value = ""

def _auto_enable_gen_range(change):
    if not gen_range_on.value:
        gen_range_on.value = True

gen_range_on.observe(_toggle_gen_range, names="value")
gen_min.observe(_auto_enable_gen_range, names="value")
gen_max.observe(_auto_enable_gen_range, names="value")

def _set_gen_msg(text, ok=False):
    color = "#166534" if ok else "#c0392b"
    gen_msg.value = '<div style="color:%s;font-size:13px;margin:4px 0">%s</div>' % (color, text)

def _draw_x_sample(rng, k, bounded, lo=None, hi=None):
    if bounded:
        return rng.uniform(lo, hi, size=k)
    return rng.standard_normal(size=k)

def _sample_unique_x(rng, m, bounded, lo=None, hi=None):
    """Draw m distinct float64 x values; %.17g text serialization round-trips them exactly."""
    xs = np.asarray(_draw_x_sample(rng, m, bounded, lo, hi), dtype=float)
    for _ in range(50):
        _, keep_idx = np.unique(xs, return_index=True)
        if len(keep_idx) == m:
            return xs, ""
        duplicate_mask = np.ones(m, dtype=bool)
        duplicate_mask[keep_idx] = False
        xs[duplicate_mask] = _draw_x_sample(
            rng, int(duplicate_mask.sum()), bounded, lo, hi)
    return None, ("The current interval is too narrow to provide %d distinct x values; "
                  "widen Min–Max or reduce the point count." % m)

def generate_random_points(_):
    raw_count = (gen_count.value or "").strip()
    if raw_count == "":
        _set_gen_msg("Please enter the number of points before generating.")
        return
    try:
        m = int(raw_count)
    except Exception:
        _set_gen_msg("Point count must be an integer.")
        return
    if m < 2:
        _set_gen_msg("Point count must be an integer ≥ 2 (Mode 2 needs at least two points).")
        return
    if m > 20000:
        _set_gen_msg("At most 20000 points per generation; use batches or paste data directly.")
        return
    rng = np.random.default_rng()
    if gen_range_on.value:
        lo, hi = float(gen_min.value), float(gen_max.value)
        if not (np.isfinite(lo) and np.isfinite(hi)):
            _set_gen_msg("Min and Max must both be valid numbers.")
            return
        if lo == hi:
            _set_gen_msg("Min equals Max; generation is blocked. Require Min strictly smaller than Max.")
            return
        if lo > hi:
            _set_gen_msg("Min is greater than Max (left > right); generation is blocked. Use Min < Max.")
            return
        xs, x_error = _sample_unique_x(rng, m, True, lo, hi)
        if x_error:
            _set_gen_msg(x_error)
            return
        ys = rng.uniform(lo, hi, size=m)
        range_text = "x and y uniform on [%.6g, %.6g]" % (lo, hi)
    else:
        xs, x_error = _sample_unique_x(rng, m, False)
        if x_error:
            _set_gen_msg(x_error)
            return
        ys = rng.standard_normal(size=m)
        range_text = "x and y from standard normal N(0,1) (range not limited)"
    order = gen_order.value
    pts = list(zip(xs.tolist(), ys.tolist()))
    if order == "asc":
        pts.sort(key=lambda p: p[0])
    elif order == "desc":
        pts.sort(key=lambda p: p[0], reverse=True)
    points_area.value = "\n".join("%.17g, %.17g" % (x, y) for x, y in pts)
    order_text = {"asc": "filled in ascending x order",
                  "desc": "filled in descending x order (computation still sorts ascending)",
                  "none": "kept in random order"}[order]
    _set_gen_msg("Generated %d points: %s; %s. Review them, then click Compute full polynomial."
                 % (m, range_text, order_text), ok=True)

gen_btn.on_click(generate_random_points)
gen_ui = widgets.VBox([
    widgets.HTML("<b>Random test-point generator (fills the textarea above only; no auto-run)</b>"),
    widgets.HBox([gen_count, gen_btn]),
    gen_range_on,
    widgets.HBox([gen_min, gen_max]),
    gen_order,
    gen_hint,
    gen_msg
], layout=widgets.Layout(border="1px dashed #94a3b8", padding="10px 12px",
                         margin="8px 0", max_width="760px"))

btn2 = widgets.Button(description='Compute full polynomial', button_style='primary')
btn2.on_click(run_mode2)

mode2_ui = widgets.VBox([
    conv_select2,
    widgets.Label('One point per line as x, y (space or comma separated)'),
    points_area,
    gen_ui,
    order_box,
    order_hint2,
    prog2_box,
    btn2,
    mode2_out
])

tab = widgets.Tab(children=[mode1_ui, mode2_ui])
tab.titles = ('Mode 1: h known', 'Mode 2: data points')
display(tab)

In [5]:
# ===== Cell 6 | Offline self-check: reference examples for both conventions; rerun anytime =====
# --- Physicists: h=x^2+2x+3 -> F(7/2,-1,1/4) ---
alpha1, beta1, h1 = exact_expansion(
    [sp.Rational(3), sp.Rational(2), sp.Rational(1)], 'phys')
assert beta1 == [sp.Rational(7, 2), sp.Rational(-1), sp.Rational(1, 4)]
assert alpha1 == [sp.Rational(7, 2), sp.Rational(1), sp.Rational(1, 4)]

# --- Probabilists: same function -> F_He(4,2,1), Rodrigues basis (4,-2,1) ---
alpha1p, beta1p, _ = exact_expansion(
    [sp.Rational(3), sp.Rational(2), sp.Rational(1)], 'prob')
assert alpha1p == [sp.Rational(4), sp.Rational(2), sp.Rational(1)]
assert beta1p == [sp.Rational(4), sp.Rational(-2), sp.Rational(1)]

# --- 9x^4 example: both conventions reconstruct exactly ---
coeffs4 = [sp.Rational(3), sp.Rational(-11), sp.Rational(6),
           sp.Rational(-7), sp.Rational(9)]
for conv in ('phys', 'prob'):
    a2, _, h2 = exact_expansion(coeffs4, conv)
    rebuilt = sp.expand(sum(a2[n] * exact_basis_poly(n, conv) for n in range(5)))
    assert rebuilt == h2

# --- Dense-point trapezoidal rule: both conventions approach the exact coefficients (grid +/-5; probabilists' tail is heavier) ---
xg = np.linspace(-5, 5, 501)
yg = xg ** 2 + 2 * xg + 3
al_phys, _ = trapezoid_alpha(xg, yg, 2, 'phys')
err_phys = float(np.abs(al_phys - np.array([3.5, 1.0, 0.25])).max())
assert err_phys < 1e-3
al_prob, _ = trapezoid_alpha(xg, yg, 2, 'prob')
err_prob = float(np.abs(al_prob - np.array([4.0, 2.0, 1.0])).max())
assert err_prob < 1e-3

# --- Three-tier order warning logic ---
assert order_warning(6, 4)[0] == 'caution'     # 5 parameters from 6 points
assert order_warning(6, 9)[0] == 'danger'      # more parameters than points, underdetermined
assert order_warning(101, 20)[0] == 'ok'       # dense points, no warning
assert warning_banner(101, 20) == ''

# --- First differences and SSE checks (dense grid, physicists) ---
d_phys = np.diff(al_phys)
assert np.allclose([d_phys[n - 1] for n in range(1, len(al_phys))],
                   [al_phys[n] - al_phys[n - 1] for n in range(1, len(al_phys))])
assert np.isclose(d_phys.sum(), al_phys[-1] - al_phys[0])   # telescoping sum
P_g = basis_values(xg, 2, 'phys')
yfit_g = sum(al_phys[n] * P_g[n] for n in range(3))
sse_g = float(np.sum((yg - yfit_g) ** 2))
assert sse_g >= 0

# --- Classroom Truncation Rule: node-trapezoid Delta_n reproduces whiteboard values (2026-11-19) ---
x6 = np.array([0.13, 0.47, 1.26, 1.83, 3.14, 4.72])
y6 = np.array([2.417, 3.186, 4.732, 4.105, 2.891, 1.634])
a6, Pv6 = trapezoid_alpha(x6, y6, 5, 'phys')
d6 = trapezoid_delta(x6, Pv6, a6, 'phys')
for _n, _v in {1: 0.506050, 2: 0.123712, 3: 0.092488,
               4: 0.042527, 5: 0.021085}.items():
    assert abs(d6[_n] - _v) < 5e-6, (_n, d6[_n])
assert d6[3] >= 0.05 and d6[4] < 0.05
assert stop_banner(d6, 4, 'caution') != ''
assert stop_banner(d6, 5, 'danger') != ''
assert stop_banner(d6, 99, 'ok') == ''
assert r'\Delta_n' in static_delta_tex('phys')
assert r'He_n' in static_delta_tex('prob')

# --- 20,000-point generator regression: high-precision text must not create duplicate x ---
_rng_gen_test = np.random.default_rng(20261119)
_x_range, _x_range_err = _sample_unique_x(_rng_gen_test, 20000, True, -2.0, 2.0)
assert _x_range_err == ""
assert len(np.unique(_x_range)) == 20000
assert len(set("%.17g" % v for v in _x_range)) == 20000
_x_normal, _x_normal_err = _sample_unique_x(_rng_gen_test, 20000, False)
assert _x_normal_err == ""
assert len(np.unique(_x_normal)) == 20000

print('Self-check passed:')
print('  1. Physicists: h=x^2+2x+3 -> F(7/2, -1, 1/4), matching the handwritten note')
print('  2. Probabilists: h=x^2+2x+3 -> F_He(4, 2, 1), Rodrigues basis (4, -2, 1)')
print('  3. The 9x^4 example is reconstructed exactly under both conventions')
print('  4. Dense-point max error: physicists %.2e, probabilists %.2e (both < 1e-3)'
      % (err_phys, err_prob))
print('  5. Three-tier order warning (caution/danger/ok) is correct; high orders are no longer blocked')
print('  6. First differences satisfy the telescoping sum and SSE works (dense-grid SSE=%.2e)' % sse_g)
print('  7. Classroom Delta_n (node-trapezoid norms) matches the whiteboard; 0.05 stop hint works')
print('  8. The 20,000-point generator keeps x unique and round-trip parseable in both range/normal modes')
display(Latex(
    r'h(x)=x^2+2x+3=\frac{7}{2}H_0+H_1+\frac{1}{4}H_2'
    r'=4He_0+2He_1+He_2'))

Self-check passed:
  1. Physicists: h=x^2+2x+3 -> F(7/2, -1, 1/4), matching the handwritten note
  2. Probabilists: h=x^2+2x+3 -> F_He(4, 2, 1), Rodrigues basis (4, -2, 1)
  3. The 9x^4 example is reconstructed exactly under both conventions
  4. Dense-point max error: physicists 6.88e-11, probabilists 2.98e-05 (both < 1e-3)
  5. Three-tier order warning (caution/danger/ok) is correct; high orders are no longer blocked
  6. First differences satisfy the telescoping sum and SSE works (dense-grid SSE=4.56e-15)
  7. Classroom Delta_n (node-trapezoid norms) matches the whiteboard; 0.05 stop hint works
  8. The 20,000-point generator keeps x unique and round-trip parseable in both range/normal modes


<IPython.core.display.Latex object>